In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

In [2]:
loader = TextLoader('./rag.txt')
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20
)
chunks = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
db = Chroma.from_documents(chunks, embeddings)

retriever = db.as_retriever()

llm = ChatOllama(model='gemma')

map_prompt = PromptTemplate(
    template="""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"],
)

# REDUCE prompt (combine answers)
combine_prompt = PromptTemplate(
    template="""
Given the following answers, create a final answer.

Answers:
{summaries}

Final Answer:
""",
    input_variables=["summaries"],
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="map_reduce",
    chain_type_kwargs={
        "question_prompt": map_prompt,
        "combine_prompt": combine_prompt,
        "verbose": True
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:

query = "What is static knowledge in rag?"
result = qa.invoke(query)

print(result)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Use the following context to answer the question.

Context:
In short: The RAG system works by doing a smart lookup in a knowledge base every time a question comes in, and giving the LLM those “clues” so it can respond accurately. This dynamic retrieval and generation loop happens behind the scenes in seconds. The user just sees a helpful answer that’s backed by actual data.

Question:
What is static knowledge in rag?

Answer:

Prompt after formatting:

Use the following context to answer the question.

Context:
Summary: The LLM, vector store, and retriever form a pipeline: the retriever finds relevant info from the vector DB, and the LLM uses that info to generate a better answer. This combination is what makes a RAG system powerful.

How Does a RAG System Work? (High-Level Workflow)
To understand the RAG system architecture, let’s walk through a typical query flow in a retrie

/opt/anaconda3/lib/python3.13/site-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))




> Entering new LLMChain chain...
Prompt after formatting:

Given the following answers, create a final answer.

Answers:
The provided text does not contain any information regarding static knowledge in RAG, so I am unable to answer this question from the given context.

The provided text does not contain any information regarding the concept of "static knowledge in rag," so I am unable to answer this question from the given context.

The provided text does not contain any information regarding the concept of "static knowledge" in RAG, so I am unable to answer this question from the given context.

The provided context does not contain any information regarding static knowledge in RAG, so I am unable to answer this question from the given context.

Final Answer:


> Finished chain.

> Finished chain.
{'query': 'What is static knowledge in rag?', 'result': 'The provided text does not contain any information regarding static knowledge in RAG, so I am unable to answer this question from 

In [4]:
retriever.invoke(query)

[Document(metadata={'source': './rag.txt'}, page_content='In short: The RAG system works by doing a smart lookup in a knowledge base every time a question comes in, and giving the LLM those “clues” so it can respond accurately. This dynamic retrieval and generation loop happens behind the scenes in seconds. The user just sees a helpful answer that’s backed by actual data.'),
 Document(metadata={'source': './rag.txt'}, page_content='Summary: The LLM, vector store, and retriever form a pipeline: the retriever finds relevant info from the vector DB, and the LLM uses that info to generate a better answer. This combination is what makes a RAG system powerful.\n\nHow Does a RAG System Work? (High-Level Workflow)\nTo understand the RAG system architecture, let’s walk through a typical query flow in a retrieval-augmented generation system:'),
 Document(metadata={'source': './rag.txt'}, page_content='RAG addresses both issues. By connecting the LLM to an external vector database of documents or